# [SQL 실습 #05] MySQL 조건 검색 더 깊게 보기
## WHERE, BETWEEN, IS NULL, NOT, LIKE 응용

> **학생용 실습 노트북 - TODO 완성형**

지난 시간에는 `SELECT`를 이용해 저장된 데이터를 조회하는 기본 방법을 배웠습니다.  
이번 시간에는 `WHERE`를 중심으로 조건 검색을 조금 더 깊게 연습합니다.

### 오늘의 학습 목표
1. WHERE 기본 문법 복습
2. 비교 연산자 다시 정리
3. BETWEEN으로 범위 검색
4. NOT으로 조건 반대로 검색
5. NOT IN으로 여러 값 제외
6. IS NULL로 비어 있는 값 찾기
7. IS NOT NULL로 값이 있는 데이터 찾기
8. LIKE로 문자열 검색 응용
9. AND와 OR를 함께 쓸 때 괄호 사용

> **핵심:** `WHERE`는 테이블에서 원하는 조건에 맞는 데이터만 골라내는 SQL 필터입니다.

# 1. 실습 환경 준비

PDF에서는 이전 시간에 만든 `school` 데이터베이스를 이어서 사용합니다.  
하지만 Google Colab은 런타임이 초기화되면 MySQL과 데이터가 사라질 수 있으므로,
이 노트북에서는 같은 구조의 실습 환경을 자동으로 준비합니다.

아래 셀은 수정하지 말고 실행하세요.

In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start
print("✅ MySQL 서버 준비 완료")

# 2. 조건 검색용 실습 데이터 준비

PDF #05의 실습 흐름에 맞춰 다음 10개 데이터를 사용합니다.

| id | name | grade | class_name |
|---:|---|---:|---|
| 1 | 김부장 | 3 | 컴퓨터과 |
| 2 | 혼이 | 1 | 영혼반 |
| 3 | 라즈베리 | 2 | 임베디드반 |
| 4 | 리눅스 | 3 | 서버반 |
| 5 | 파이썬 | 1 | 프로그래밍반 |
| 6 | 김코딩 | 1 | 컴퓨터과 |
| 7 | 데이터왕 | 2 | 데이터베이스반 |
| 8 | 홍길동 | 2 | 컴퓨터과 |
| 9 | 이몽룡 | 3 | 데이터베이스반 |
| 10 | 정보없음 | NULL | NULL |

`정보없음`은 `NULL` 검색을 연습하기 위해 학년과 학과를 비워 둡니다.

PDF의 날짜 범위 예제도 그대로 연습할 수 있도록 `created_at`을
`2026-08-01 19:00:00`으로 고정합니다.

In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, created_at)
VALUES
('김부장', 3, '컴퓨터과', '2026-08-01 19:00:00'),
('혼이', 1, '영혼반', '2026-08-01 19:00:00'),
('라즈베리', 2, '임베디드반', '2026-08-01 19:00:00'),
('리눅스', 3, '서버반', '2026-08-01 19:00:00'),
('파이썬', 1, '프로그래밍반', '2026-08-01 19:00:00'),
('김코딩', 1, '컴퓨터과', '2026-08-01 19:00:00'),
('데이터왕', 2, '데이터베이스반', '2026-08-01 19:00:00'),
('홍길동', 2, '컴퓨터과', '2026-08-01 19:00:00'),
('이몽룡', 3, '데이터베이스반', '2026-08-01 19:00:00');

INSERT INTO students (name, created_at)
VALUES ('정보없음', '2026-08-01 19:00:00');
SQL

echo "✅ school.students 실습 데이터 준비 완료"

# 3. SQL 실행 도우미와 자동점검

- SQL에 `TODO`가 남아 있으면 실행하지 않습니다.
- MySQL 오류를 화면에 표시합니다.
- 각 TODO 결과를 정답 결과와 비교합니다.
- 조회 결과가 같으면 문법 표현이 조금 달라도 통과할 수 있습니다.
- 학생 TODO 셀에서는 `DROP`, `TRUNCATE`, `DELETE`, `UPDATE`, `ALTER`를 차단합니다.

> 이 셀은 수정하지 마세요.

In [ ]:
import subprocess
import textwrap
import base64
import re

LAST_SQL = ""
LAST_SQL_OK = False
TODO_STATUS = {}

def _execute_mysql(sql):
    result = subprocess.run(
        ["sudo", "mysql", "--batch", "--raw"],
        input=sql,
        text=True,
        capture_output=True
    )
    return result.returncode == 0, result.stdout.strip(), result.stderr.strip()

def run_sql(sql):
    global LAST_SQL, LAST_SQL_OK

    sql = textwrap.dedent(sql).strip()
    LAST_SQL = sql
    LAST_SQL_OK = False

    real_lines = [
        line for line in sql.splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ]
    real_sql = "\n".join(real_lines).strip()

    if not real_sql:
        print("⚠️ 아직 SQL이 작성되지 않았습니다.")
        return False

    if "TODO" in real_sql:
        print("⚠️ SQL 안에 TODO가 남아 있습니다. 먼저 완성하세요.")
        return False

    forbidden = [r"\bDROP\b", r"\bTRUNCATE\b", r"\bDELETE\b", r"\bUPDATE\b", r"\bALTER\b"]
    if any(re.search(p, real_sql, re.IGNORECASE) for p in forbidden):
        print("❌ 이 TODO 실습에서는 데이터 변경/삭제 명령을 실행하지 않습니다.")
        return False

    ok, out, err = _execute_mysql(sql)

    if out:
        print(out)

    if not ok:
        print("❌ MySQL 오류가 발생했습니다.")
        print(err)
        return False

    LAST_SQL_OK = True
    print("✅ SQL 실행 완료")
    return True

_EXPECTED_B64 = {1: 'VVNFIHNjaG9vbDsKU0hPVyBUQUJMRVM7', 2: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA9IDM7', 3: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA+PSAyOw==', 4: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSAhPSAxOw==', 5: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBCRVRXRUVOIDEgQU5EIDI7', 6: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjcmVhdGVkX2F0IEJFVFdFRU4gJzIwMjYtMDgtMDEgMDA6MDA6MDAnIEFORCAnMjAyNi0wOC0wMSAyMzo1OTo1OSc7', 7: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBOT1QgZ3JhZGUgPSAxOw==', 8: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBOT1QgSU4gKDEsIDIpOw==', 9: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBJUyBOVUxMOw==', 10: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBJUyBOT1QgTlVMTDs=', 11: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIElTIE5PVCBOVUxMOw==', 12: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lIExJS0UgJ+q5gCUnOw==', 13: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lIExJS0UgJyXquYAlJzs=', 14: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIExJS0UgJyXrsJglJzs=', 15: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIE5PVCBMSUtFICcl67CYJSc7', 16: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIE5PVCBMSUtFICcl67CYJScgT1IgY2xhc3NfbmFtZSBJUyBOVUxMOw==', 17: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSAoZ3JhZGUgPSAxIE9SIGdyYWRlID0gMikgQU5EIGNsYXNzX25hbWUgPSAn7Lu07ZOo7YSw6rO8Jzs=', 18: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA9IDI7', 19: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSA+PSAyOw==', 20: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBCRVRXRUVOIDEgQU5EIDI7', 21: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBncmFkZSBJUyBOVUxMOw==', 22: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIElTIE5PVCBOVUxMOw==', 23: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBuYW1lIExJS0UgJ+q5gCUnOw==', 24: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSBjbGFzc19uYW1lIExJS0UgJyXrjbDsnbTthLAlJzs=', 25: 'VVNFIHNjaG9vbDsKU0VMRUNUICogRlJPTSBzdHVkZW50cyBXSEVSRSAoZ3JhZGUgPSAxIE9SIGdyYWRlID0gMikgQU5EIGNsYXNzX25hbWUgPSAn7Lu07ZOo7YSw6rO8Jzs='}

def _decode_expected(todo_id):
    return base64.b64decode(_EXPECTED_B64[todo_id]).decode("utf-8")

def _normalize_output(text, todo_id):
    lines = [line.rstrip() for line in text.strip().splitlines() if line.strip()]
    if not lines:
        return []
    if todo_id == 1:
        return lines
    header = lines[0]
    rows = sorted(lines[1:])
    return [header] + rows

def check_todo(todo_id, sql, current_todo_id=None):
    global TODO_STATUS

    print(f"🔎 TODO {todo_id} 자동점검")

    if current_todo_id != todo_id:
        TODO_STATUS[todo_id] = False
        print("❌ 바로 위 TODO 코드 셀을 먼저 실행하세요.")
        return False

    student_sql = textwrap.dedent(sql).strip()

    if not student_sql:
        TODO_STATUS[todo_id] = False
        print("❌ SQL이 비어 있습니다.")
        return False

    if "TODO" in student_sql:
        TODO_STATUS[todo_id] = False
        print("❌ TODO가 아직 남아 있습니다.")
        return False

    ok_s, out_s, err_s = _execute_mysql(student_sql)
    if not ok_s:
        TODO_STATUS[todo_id] = False
        print("❌ SQL 실행 오류가 있습니다.")
        print(err_s)
        return False

    expected_sql = _decode_expected(todo_id)
    ok_e, out_e, err_e = _execute_mysql(expected_sql)
    if not ok_e:
        TODO_STATUS[todo_id] = False
        print("⚠️ 자동점검 기준 실행 오류입니다. 교사에게 알려주세요.")
        return False

    passed = (_normalize_output(out_s, todo_id) == _normalize_output(out_e, todo_id))
    TODO_STATUS[todo_id] = passed

    if passed:
        print("✅ 통과: 요구한 결과가 정확합니다.")
    else:
        print("❌ 재확인: 결과가 요구사항과 다릅니다.")
        print("   → WHERE 조건, NULL, LIKE, 괄호를 다시 확인하세요.")
    return passed

def show_todo_progress(total=25):
    done = sum(TODO_STATUS.get(i) is True for i in range(1, total + 1))
    failed = sum(TODO_STATUS.get(i) is False for i in range(1, total + 1))
    unchecked = total - done - failed

    print("=" * 48)
    print("📊 TODO 자동점검 진행 현황")
    print("=" * 48)

    for start in range(1, total + 1, 5):
        end = min(start + 4, total)
        parts = []
        for i in range(start, end + 1):
            state = TODO_STATUS.get(i)
            mark = "✅" if state is True else "❌" if state is False else "⬜"
            parts.append(f"{i}:{mark}")
        print("   ".join(parts))

    print("-" * 48)
    print(f"통과       : {done}/{total}")
    print(f"재확인 필요: {failed}/{total}")
    print(f"미점검     : {unchecked}/{total}")
    print(f"진행률     : {done / total * 100:.1f}%")

# 4. 사용할 데이터베이스 선택하기

`school` 데이터베이스를 선택하고 테이블 목록을 확인하세요.

---
## TODO 1
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE TODO;
SHOW TODO;
"""
_todo_id = 1
run_sql(sql)

In [ ]:
# TODO 1 자동점검 - 수정하지 마세요.
check_todo(1, sql, globals().get('_todo_id'))

# 5. WHERE 기본 복습하기

`WHERE`는 조건에 맞는 데이터만 조회할 때 사용합니다.

```sql
SELECT 컬럼명 FROM 테이블명 WHERE 조건;
```

**3학년 학생**을 전체 컬럼으로 조회하세요.

---
## TODO 2
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE TODO = TODO;
"""
_todo_id = 2
run_sql(sql)

In [ ]:
# TODO 2 자동점검 - 수정하지 마세요.
check_todo(2, sql, globals().get('_todo_id'))

# 6. 비교 연산자 — 2학년 이상

비교 연산자에는 `=`, `!=`, `<>`, `>`, `>=`, `<`, `<=`가 있습니다.

**2학년 이상** 학생을 조회하세요.

---
## TODO 3
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO;
"""
_todo_id = 3
run_sql(sql)

In [ ]:
# TODO 3 자동점검 - 수정하지 마세요.
check_todo(3, sql, globals().get('_todo_id'))

# 7. 비교 연산자 — 1학년이 아닌 학생

`!=`와 `<>`는 모두 **같지 않다**는 뜻입니다.

`!=`를 사용해 **1학년이 아닌 학생**을 조회하세요.

> `NULL`은 일반 비교 연산으로 찾을 수 없습니다.

---
## TODO 4
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO;
"""
_todo_id = 4
run_sql(sql)

In [ ]:
# TODO 4 자동점검 - 수정하지 마세요.
check_todo(4, sql, globals().get('_todo_id'))

# 8. BETWEEN으로 범위 검색하기

`BETWEEN 1 AND 2`는 **1 이상 2 이하**이며 양 끝값을 모두 포함합니다.

1학년부터 2학년까지 조회하세요.

---
## TODO 5
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO TODO TODO;
"""
_todo_id = 5
run_sql(sql)

In [ ]:
# TODO 5 자동점검 - 수정하지 마세요.
check_todo(5, sql, globals().get('_todo_id'))

# 9. BETWEEN 날짜 검색 예시

PDF의 예시처럼 `2026-08-01` 하루 동안 입력된 데이터를 조회하세요.

시작: `2026-08-01 00:00:00`  
끝: `2026-08-01 23:59:59`

---
## TODO 6
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE created_at TODO 'TODO'
                     TODO 'TODO';
"""
_todo_id = 6
run_sql(sql)

In [ ]:
# TODO 6 자동점검 - 수정하지 마세요.
check_todo(6, sql, globals().get('_todo_id'))

# 10. NOT으로 조건 반대로 검색하기

`NOT`은 조건을 반대로 만듭니다.

`NOT`을 사용하여 **1학년이 아닌 학생**을 조회하세요.

---
## TODO 7
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE TODO grade = TODO;
"""
_todo_id = 7
run_sql(sql)

In [ ]:
# TODO 7 자동점검 - 수정하지 마세요.
check_todo(7, sql, globals().get('_todo_id'))

# 11. NOT IN 사용하기

여러 값에 해당하지 않는 데이터를 찾을 때 `NOT IN`을 사용합니다.

1학년과 2학년을 제외한 학생을 조회하세요.

---
## TODO 8
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO (TODO, TODO);
"""
_todo_id = 8
run_sql(sql)

In [ ]:
# TODO 8 자동점검 - 수정하지 마세요.
check_todo(8, sql, globals().get('_todo_id'))

# 12. IS NULL로 비어 있는 값 찾기

`NULL`은 `0`이나 빈 문자열과 다릅니다.

잘못된 방식:
```sql
WHERE grade = NULL;
```

올바른 방식은 `IS NULL`입니다.

학년 정보가 없는 학생을 조회하세요.

---
## TODO 9
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO;
"""
_todo_id = 9
run_sql(sql)

In [ ]:
# TODO 9 자동점검 - 수정하지 마세요.
check_todo(9, sql, globals().get('_todo_id'))

# 13. IS NOT NULL로 값이 있는 데이터 찾기 — 학년

학년 정보가 입력되어 있는 학생만 조회하세요.

---
## TODO 10
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO TODO;
"""
_todo_id = 10
run_sql(sql)

In [ ]:
# TODO 10 자동점검 - 수정하지 마세요.
check_todo(10, sql, globals().get('_todo_id'))

# 14. IS NOT NULL로 값이 있는 데이터 찾기 — 학과

학과 정보가 입력되어 있는 학생만 조회하세요.

---
## TODO 11
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO TODO TODO;
"""
_todo_id = 11
run_sql(sql)

In [ ]:
# TODO 11 자동점검 - 수정하지 마세요.
check_todo(11, sql, globals().get('_todo_id'))

# 15. LIKE로 문자 검색 — 김으로 시작

`%`는 0개 이상의 아무 문자를 의미합니다.

- `'김%'` : 김으로 시작
- `'%김'` : 김으로 끝남
- `'%김%'` : 김이 포함됨

이름이 **김으로 시작하는 학생**을 조회하세요.

---
## TODO 12
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name TODO 'TODO';
"""
_todo_id = 12
run_sql(sql)

In [ ]:
# TODO 12 자동점검 - 수정하지 마세요.
check_todo(12, sql, globals().get('_todo_id'))

# 16. LIKE로 문자 검색 — 김이 포함

이름에 **김이 포함된 학생**을 조회하세요.

---
## TODO 13
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name TODO 'TODO';
"""
_todo_id = 13
run_sql(sql)

In [ ]:
# TODO 13 자동점검 - 수정하지 마세요.
check_todo(13, sql, globals().get('_todo_id'))

# 17. LIKE로 학과명 검색하기

학과명에 **반**이 포함된 데이터를 조회하세요.

---
## TODO 14
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO 'TODO';
"""
_todo_id = 14
run_sql(sql)

In [ ]:
# TODO 14 자동점검 - 수정하지 마세요.
check_todo(14, sql, globals().get('_todo_id'))

# 18. LIKE와 NOT LIKE

학과명에 **반이 들어가지 않은 학생**을 조회하세요.

> `class_name`이 NULL인 행은 `NOT LIKE`만으로는 포함되지 않습니다.

---
## TODO 15
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO TODO 'TODO';
"""
_todo_id = 15
run_sql(sql)

In [ ]:
# TODO 15 자동점검 - 수정하지 마세요.
check_todo(15, sql, globals().get('_todo_id'))

# 19. NOT LIKE와 NULL을 함께 고려하기

다음을 모두 조회하세요.

- 학과명에 `반`이 포함되지 않은 데이터
- 학과 정보가 `NULL`인 데이터

---
## TODO 16
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO TODO 'TODO'
   TODO class_name TODO TODO;
"""
_todo_id = 16
run_sql(sql)

In [ ]:
# TODO 16 자동점검 - 수정하지 마세요.
check_todo(16, sql, globals().get('_todo_id'))

# 20. AND와 OR를 함께 사용할 때 괄호 쓰기

SQL에서는 `AND`가 `OR`보다 먼저 처리됩니다.

**1학년 또는 2학년 중에서 컴퓨터과 학생**을 조회하도록 괄호를 사용하세요.

---
## TODO 17
아래 `TODO`를 알맞게 완성하세요.

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE (grade = TODO OR grade = TODO)
  TODO class_name = 'TODO';
"""
_todo_id = 17
run_sql(sql)

In [ ]:
# TODO 17 자동점검 - 수정하지 마세요.
check_todo(17, sql, globals().get('_todo_id'))

# 21. 괄호 없는 조건과 괄호 있는 조건 비교

다음 두 SQL은 결과가 달라질 수 있습니다.

### 괄호 없음
```sql
WHERE grade = 1
   OR grade = 2
  AND class_name = '컴퓨터과';
```

### 괄호 있음
```sql
WHERE (grade = 1 OR grade = 2)
  AND class_name = '컴퓨터과';
```

아래 셀을 실행해 차이를 직접 확인하세요.

In [ ]:
sql_without = """
USE school;
SELECT id, name, grade, class_name
FROM students
WHERE grade = 1
   OR grade = 2
  AND class_name = '컴퓨터과';
"""
ok, out, err = _execute_mysql(sql_without)
print("===== ① 괄호 없음 =====")
print(out if ok else err)

sql_with = """
USE school;
SELECT id, name, grade, class_name
FROM students
WHERE (grade = 1 OR grade = 2)
  AND class_name = '컴퓨터과';
"""
ok, out, err = _execute_mysql(sql_with)
print("\n===== ② 괄호 있음 =====")
print(out if ok else err)

print("\n💡 두 결과가 왜 다른지 설명해 보세요.")

# 22. 조건 검색 실습 문제

PDF의 8개 실습 문제를 직접 풀어봅니다.

## 문제 1. 2학년 학생만 조회하기

2학년 학생만 조회하세요.

### TODO 18

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE TODO = TODO;
"""
_todo_id = 18
run_sql(sql)

In [ ]:
# TODO 18 자동점검 - 수정하지 마세요.
check_todo(18, sql, globals().get('_todo_id'))

## 문제 2. 2학년 이상 학생 조회하기

2학년 이상 학생을 조회하세요.

### TODO 19

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO;
"""
_todo_id = 19
run_sql(sql)

In [ ]:
# TODO 19 자동점검 - 수정하지 마세요.
check_todo(19, sql, globals().get('_todo_id'))

## 문제 3. 1학년부터 2학년까지 조회하기

`BETWEEN`을 사용하세요.

### TODO 20

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO TODO TODO;
"""
_todo_id = 20
run_sql(sql)

In [ ]:
# TODO 20 자동점검 - 수정하지 마세요.
check_todo(20, sql, globals().get('_todo_id'))

## 문제 4. 학년 정보가 없는 학생 조회하기

`IS NULL`을 사용하세요.

### TODO 21

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE grade TODO TODO;
"""
_todo_id = 21
run_sql(sql)

In [ ]:
# TODO 21 자동점검 - 수정하지 마세요.
check_todo(21, sql, globals().get('_todo_id'))

## 문제 5. 학과 정보가 있는 학생만 조회하기

`IS NOT NULL`을 사용하세요.

### TODO 22

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO TODO TODO;
"""
_todo_id = 22
run_sql(sql)

In [ ]:
# TODO 22 자동점검 - 수정하지 마세요.
check_todo(22, sql, globals().get('_todo_id'))

## 문제 6. 이름이 김으로 시작하는 학생 조회하기

`LIKE`와 `%`를 사용하세요.

### TODO 23

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE name TODO 'TODO';
"""
_todo_id = 23
run_sql(sql)

In [ ]:
# TODO 23 자동점검 - 수정하지 마세요.
check_todo(23, sql, globals().get('_todo_id'))

## 문제 7. 학과명에 데이터가 들어가는 학생 조회하기

`데이터`라는 글자가 포함된 학과를 찾으세요.

### TODO 24

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE class_name TODO 'TODO';
"""
_todo_id = 24
run_sql(sql)

In [ ]:
# TODO 24 자동점검 - 수정하지 마세요.
check_todo(24, sql, globals().get('_todo_id'))

## 문제 8. 1학년 또는 2학년 중 컴퓨터과 학생 조회하기

괄호를 사용하여 `1학년 또는 2학년` 조건을 먼저 묶으세요.

### TODO 25

In [ ]:
sql = """
USE school;
SELECT * FROM students
WHERE (grade = TODO OR grade = TODO)
  TODO class_name = 'TODO';
"""
_todo_id = 25
run_sql(sql)

In [ ]:
# TODO 25 자동점검 - 수정하지 마세요.
check_todo(25, sql, globals().get('_todo_id'))

# 23. 자주 발생하는 실수

## ① NULL을 `=`로 비교
잘못된 예:
```sql
WHERE grade = NULL;
```

올바른 예:
```sql
WHERE grade IS NULL;
```

## ② 문자열에 작은따옴표를 쓰지 않음
잘못된 예:
```sql
WHERE class_name = 컴퓨터과;
```

올바른 예:
```sql
WHERE class_name = '컴퓨터과';
```

## ③ LIKE에서 `%` 위치를 헷갈림
```text
'김%'   → 김으로 시작
'%김'   → 김으로 끝남
'%김%'  → 김이 포함됨
```

## ④ AND와 OR를 함께 쓰면서 괄호를 생략
조건이 복잡해질수록 괄호를 사용해 의도를 분명하게 표현하세요.

# 24. 오늘 실습 코드 전체 정리

```sql
USE school;

SELECT * FROM students WHERE grade = 3;
SELECT * FROM students WHERE grade >= 2;
SELECT * FROM students WHERE grade != 1;
SELECT * FROM students WHERE grade BETWEEN 1 AND 2;

SELECT * FROM students
WHERE created_at BETWEEN '2026-08-01 00:00:00'
                     AND '2026-08-01 23:59:59';

SELECT * FROM students WHERE NOT grade = 1;
SELECT * FROM students WHERE grade NOT IN (1, 2);

SELECT * FROM students WHERE grade IS NULL;
SELECT * FROM students WHERE grade IS NOT NULL;
SELECT * FROM students WHERE class_name IS NOT NULL;

SELECT * FROM students WHERE name LIKE '김%';
SELECT * FROM students WHERE name LIKE '%김%';
SELECT * FROM students WHERE class_name LIKE '%반%';

SELECT * FROM students WHERE class_name NOT LIKE '%반%';

SELECT * FROM students
WHERE class_name NOT LIKE '%반%'
   OR class_name IS NULL;

SELECT * FROM students
WHERE (grade = 1 OR grade = 2)
  AND class_name = '컴퓨터과';
```

# 25. 전체 TODO 자동점검 결과

- ✅ 통과
- ❌ 재확인 필요
- ⬜ 미점검

In [ ]:
show_todo_progress(25)

print("\n===== 실습 데이터 기본 상태 확인 =====")
checks = [
    ("전체 행 개수", "SELECT COUNT(*) FROM school.students;"),
    ("grade IS NULL", "SELECT COUNT(*) FROM school.students WHERE grade IS NULL;"),
    ("class_name IS NULL", "SELECT COUNT(*) FROM school.students WHERE class_name IS NULL;"),
]
for label, query in checks:
    r = subprocess.run(["sudo", "mysql", "-N", "-B", "-e", query], text=True, capture_output=True)
    print(f"{label}: {r.stdout.strip()}")

# 26. 오늘 배운 내용 정리

```text
WHERE       → 원하는 조건에 맞는 데이터만 조회
BETWEEN     → 범위 필터
NOT         → 조건을 반대로
NOT IN      → 여러 값 제외
IS NULL     → 값이 비어 있는 상태 확인
IS NOT NULL → 값이 존재하는 상태 확인
LIKE        → 문자열 패턴 검색
NOT LIKE    → 특정 문자열 패턴 제외
( )         → AND와 OR가 섞인 조건의 의도를 분명하게 표현
```

> 원하는 데이터를 정확히 꺼내 보는 것, 그것이 `SELECT + WHERE`의 핵심입니다.

# 27. 자기 점검

- [ ] WHERE의 역할을 설명할 수 있다.
- [ ] 비교 연산자를 사용할 수 있다.
- [ ] BETWEEN의 양 끝값이 포함됨을 알고 있다.
- [ ] NOT과 NOT IN을 사용할 수 있다.
- [ ] NULL과 0, 빈 문자열의 차이를 알고 있다.
- [ ] IS NULL / IS NOT NULL을 사용할 수 있다.
- [ ] LIKE의 `%` 위치에 따른 의미를 설명할 수 있다.
- [ ] NOT LIKE와 NULL의 관계를 이해한다.
- [ ] AND와 OR를 섞을 때 괄호가 필요한 이유를 설명할 수 있다.
- [ ] 복합 조건을 이용해 원하는 데이터를 조회할 수 있다.

# 28. [선택] 실습 데이터 다시 초기화하기

`RESET = True`로 바꾸면 PDF #05의 기본 실습 데이터 10개로 되돌립니다.

In [ ]:
RESET = False

if RESET:
    reset_sql = """
    USE school;
    TRUNCATE TABLE students;

    INSERT INTO students (name, grade, class_name, created_at)
    VALUES
    ('김부장', 3, '컴퓨터과', '2026-08-01 19:00:00'),
    ('혼이', 1, '영혼반', '2026-08-01 19:00:00'),
    ('라즈베리', 2, '임베디드반', '2026-08-01 19:00:00'),
    ('리눅스', 3, '서버반', '2026-08-01 19:00:00'),
    ('파이썬', 1, '프로그래밍반', '2026-08-01 19:00:00'),
    ('김코딩', 1, '컴퓨터과', '2026-08-01 19:00:00'),
    ('데이터왕', 2, '데이터베이스반', '2026-08-01 19:00:00'),
    ('홍길동', 2, '컴퓨터과', '2026-08-01 19:00:00'),
    ('이몽룡', 3, '데이터베이스반', '2026-08-01 19:00:00');

    INSERT INTO students (name, created_at)
    VALUES ('정보없음', '2026-08-01 19:00:00');
    """

    ok, out, err = _execute_mysql(reset_sql)

    if ok:
        TODO_STATUS.clear()
        print("✅ 실습 데이터를 초기화했습니다.")
        print("✅ TODO 진행 상태도 초기화했습니다.")
    else:
        print("❌ 초기화 오류")
        print(err)
else:
    print("ℹ️ RESET = False 입니다. 데이터는 변경하지 않았습니다.")

# 다음 시간 예고

## [SQL 실습 #06] MySQL 데이터 수정하기
### UPDATE와 SET 기초

다음 시간에는 이미 입력된 데이터를 수정하는 방법을 배웁니다.

```sql
UPDATE
SET
WHERE
```

특히 `UPDATE`에서 `WHERE` 조건이 정확하지 않으면 여러 행이 함께 수정될 수 있으므로,
이번 시간의 조건 검색이 매우 중요합니다.